In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from IPython.display import display


In [2]:
df = pd.read_csv("Market_Basket_Retail_Dataset.csv")

# Remove duplicate transaction-product records
df = df.drop_duplicates()

print("Rows:", len(df))
print("Transactions:", df['TransactionID'].nunique())
print("Unique Products:", df['Product'].nunique())


Rows: 2887
Transactions: 700
Unique Products: 40


In [3]:
basket = (
    df.groupby(['TransactionID', 'Product'])['Product']
      .count()
      .unstack()
      .fillna(0)
)

basket = basket.astype(bool)

display(basket.head())
print("Basket Shape:", basket.shape)

Product,Apples,Bananas,Biscuits,Bread,Butter,Cereal,Cheese,Chicken,Chips,Chocolate,...,Soap,Soft Drink,Sugar,Tea,Tissue,Tomato Sauce,Toothbrush,Toothpaste,Water,Yogurt
TransactionID,,,,,,,,,,,,,,,,,,,,,
1,False,False,False,True,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,True,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,False,False,False,True,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
5,False,False,False,False,False,False,False,True,False,True,...,False,False,False,False,True,False,False,False,False,False


Basket Shape: (700, 40)


In [4]:
frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    by='support', ascending=False
).reset_index(drop=True)

print("Number of Frequent Itemsets:", len(frequent_itemsets))
display(frequent_itemsets.head(10))


Number of Frequent Itemsets: 77


,support,itemsets
0,0.208571,frozenset({Biscuits})
1,0.150000,frozenset({Butter})
2,0.150000,frozenset({Sugar})
3,0.144286,frozenset({Bread})
4,0.141429,frozenset({Toothpaste})
5,0.131429,frozenset({Ice Cream})
6,0.130000,frozenset({Milk})
7,0.127143,frozenset({Cheese})
8,0.127143,frozenset({Pasta})
9,0.125714,frozenset({Toothbrush})


In [7]:
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.30
)

strong_rules = rules[
    (rules['confidence'] >= 0.30) &
    (rules['lift'] > 1)
].copy()

strong_rules = strong_rules.sort_values(
    by=['lift', 'confidence'], ascending=False
).reset_index(drop=True)

print("Total Rules:", len(rules))
print("Strong Rules:", len(strong_rules))
display(strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Total Rules: 98
Strong Rules: 98


,antecedents,consequents,support,confidence,lift
0,"frozenset({Cooking Oil, Rice})",frozenset({Salt}),0.051429,1.000000,10.294118
1,frozenset({Salt}),"frozenset({Cooking Oil, Rice})",0.051429,0.529412,10.294118
2,"frozenset({Onions, Chicken})",frozenset({Potatoes}),0.050000,0.945946,10.032760
3,frozenset({Potatoes}),"frozenset({Onions, Chicken})",0.050000,0.530303,10.032760
4,"frozenset({Rice, Salt})",frozenset({Cooking Oil}),0.051429,0.947368,9.897879
5,frozenset({Cooking Oil}),"frozenset({Rice, Salt})",0.051429,0.537313,9.897879
6,"frozenset({Cooking Oil, Salt})",frozenset({Rice}),0.051429,0.972973,9.459459
7,frozenset({Rice}),"frozenset({Cooking Oil, Salt})",0.051429,0.500000,9.459459
8,"frozenset({Onions, Potatoes})",frozenset({Chicken}),0.050000,0.972222,9.196697
9,frozenset({Chicken}),"frozenset({Onions, Potatoes})",0.050000,0.472973,9.196697


In [8]:
def recommend_products(product, top_n=5):
    product = product.strip()

    matches = strong_rules[
        strong_rules['antecedents'].apply(
            lambda items: product in items
        )
    ].copy()

    if matches.empty:
        return pd.DataFrame(columns=['Recommended_Product', 'Confidence', 'Lift'])

    recommendations = []
    for _, row in matches.iterrows():
        for item in row['consequents']:
            recommendations.append({
                'Recommended_Product': item,
                'Confidence': row['confidence'],
                'Lift': row['lift']
            })

    result = pd.DataFrame(recommendations)
    result = result.sort_values(
        by=['Lift', 'Confidence'], ascending=False
    ).drop_duplicates('Recommended_Product')

    return result.head(top_n).reset_index(drop=True)

# Example
recommend_products('Milk')


,Recommended_Product,Confidence,Lift
0,Cereal,0.945946,8.381800
1,Bananas,0.897436,7.138695
2,Butter,0.968750,6.458333
3,Bread,0.885714,6.138614


In [9]:
strong_rules.to_pickle("market_basket_rules.pkl")
frequent_itemsets.to_pickle("frequent_itemsets.pkl")

print("Model files saved successfully.")


Model files saved successfully.


In [15]:
import pickle

model = {
    "frequent_itemsets": frequent_itemsets,
    "rules": strong_rules
}

with open("market_basket_model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Market Basket Model created and saved successfully!")

Market Basket Model created and saved successfully!


In [17]:
import pickle

with open("market_basket_model.pkl", "rb") as file:
    model = pickle.load(file)

frequent_itemsets = model["frequent_itemsets"]
strong_rules = model["rules"]

print("Model loaded successfully!")

Model loaded successfully!


In [22]:
!pip install gradio

   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   -- ------------------------------------- 1.8/31.4 MB 10.1 MB/s eta 0:00:03
   ----- ---------------------------------- 4.2/31.4 MB 10.6 MB/s eta 0:00:03
   -------- ------------------------------- 6.3/31.4 MB 10.3 MB/s eta 0:00:03
   ----------- ---------------------------- 8.9/31.4 MB 10.8 MB/s eta 0:00:03
   ------------- -------------------------- 10.7/31.4 MB 10.5 MB/s eta 0:00:02
   --------------- ------------------------ 12.3/31.4 MB 10.0 MB/s eta 0:00:02
   ----------------- ---------------------- 13.6/31.4 MB 9.2 MB/s eta 0:00:02
   ------------------ --------------------- 14.4/31.4 MB 8.6 MB/s eta 0:00:02
   ------------------- -------------------- 14.9/31.4 MB 8.1 MB/s eta 0:00:03
   -------------------- ------------------- 15.7/31.4 MB 7.6 MB/s eta 0:00:03
   --------------------- ------------------ 16.8/31.4 MB 7.3 MB/s eta 0:00:03
   ---------------------- ----------------- 17.8/31.4 MB 7.1 MB/s eta 

In [5]:
import gradio as gr
import pandas as pd
import pickle

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("Market_Basket_Retail_Dataset.csv")

with open("market_basket_model.pkl", "rb") as f:
    model = pickle.load(f)

frequent_itemsets = model["frequent_itemsets"]
rules = model["rules"]


# =========================
# FIND PRODUCT COLUMN
# =========================

possible_columns = [
    "Item", "item",
    "Product", "product",
    "Itemname", "Item Name",
    "Description", "description"
]

product_column = None

for col in possible_columns:
    if col in df.columns:
        product_column = col
        break

if product_column is None:
    for col in df.columns:
        if df[col].dtype == "object":
            product_column = col
            break

if product_column is None:
    raise ValueError("Product column not found in dataset.")


# =========================
# GET ALL PRODUCTS
# =========================

products = sorted(
    df[product_column]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


# =========================
# DASHBOARD STATISTICS
# =========================

total_records = len(df)
total_products = len(products)
total_itemsets = len(frequent_itemsets)
total_rules = len(rules)


# Find transaction column
transaction_columns = [
    "Transaction",
    "transaction",
    "TransactionID",
    "Transaction ID",
    "Invoice",
    "InvoiceNo",
    "Invoice No"
]

transaction_column = None

for col in transaction_columns:
    if col in df.columns:
        transaction_column = col
        break

if transaction_column:
    total_transactions = df[transaction_column].nunique()
else:
    total_transactions = total_records


# =========================
# RECOMMENDATION FUNCTION
# =========================

def recommend(selected_items):

    if not selected_items:
        return pd.DataFrame({
            "Message": [
                "Please select at least one product."
            ]
        })

    selected_items = set(selected_items)

    result = rules[
        rules["antecedents"].apply(
            lambda x: selected_items.issubset(set(x))
        )
    ].copy()

    if result.empty:
        return pd.DataFrame({
            "Message": [
                "No recommendations found for the selected combination."
            ]
        })

    result = result.sort_values(
        by="confidence",
        ascending=False
    )

    result["antecedents"] = result["antecedents"].apply(
        lambda x: ", ".join(sorted(x))
    )

    result["consequents"] = result["consequents"].apply(
        lambda x: ", ".join(sorted(x))
    )

    return result[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)


# =========================
# DASHBOARD
# =========================

with gr.Blocks(title="Market Basket Analysis") as demo:

    gr.Markdown(
        """
        # 🛒 Market Basket Analysis
        ### Retail Transaction Analysis & Product Recommendation
        """
    )

    # =========================
    # TOP DETAILS
    # =========================

    with gr.Row():

        with gr.Column():
            gr.Markdown("### Transactions")
            gr.Markdown(f"# {total_transactions}")

        with gr.Column():
            gr.Markdown("### Records")
            gr.Markdown(f"# {total_records}")

        with gr.Column():
            gr.Markdown("### Products")
            gr.Markdown(f"# {total_products}")

        with gr.Column():
            gr.Markdown("### Frequent Itemsets")
            gr.Markdown(f"# {total_itemsets}")

        with gr.Column():
            gr.Markdown("### Rules")
            gr.Markdown(f"# {total_rules}")


    gr.Markdown("---")


    # =========================
    # LEFT + RIGHT SECTION
    # =========================

    with gr.Row():

        # LEFT SIDE - PRODUCTS
        with gr.Column(scale=1):

            gr.Markdown("### 🛍️ Products")

            product_selection = gr.CheckboxGroup(
                choices=products,
                label="Select one or more products"
            )

            clear_button = gr.Button(
                "Clear Selection"
            )


        # RIGHT SIDE - DATASET
        with gr.Column(scale=5):

            gr.Markdown("### 📊 Actual Dataset")

            dataset_display = gr.Dataframe(
                value=df,
                interactive=False
            )


    gr.Markdown("---")


    # =========================
    # BOTTOM - RECOMMENDATIONS
    # =========================

    gr.Markdown("## 🔎 Recommendations")

    selected_text = gr.Markdown(
        "Select products from the left panel."
    )

    recommendation_output = gr.Dataframe(
        headers=[
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ],
        interactive=False
    )


    # =========================
    # UPDATE RECOMMENDATIONS
    # =========================

    def update_recommendations(selected):

        if not selected:
            return (
                "Select products from the left panel.",
                pd.DataFrame()
            )

        text = "**Selected:** " + " + ".join(selected)

        result = recommend(selected)

        return text, result


    product_selection.change(
        fn=update_recommendations,
        inputs=product_selection,
        outputs=[
            selected_text,
            recommendation_output
        ]
    )


    # =========================
    # CLEAR BUTTON
    # =========================

    clear_button.click(
        fn=lambda: (
            [],
            "Select products from the left panel.",
            pd.DataFrame()
        ),
        inputs=None,
        outputs=[
            product_selection,
            selected_text,
            recommendation_output
        ]
    )


# =========================
# LAUNCH
# =========================

demo.launch() 

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
